<a href="https://colab.research.google.com/github/TECNOMauroo/CPSN_2026/blob/(CURSO)-Data-Science-Fundamentals-(certificaci%C3%B3n-docente-2026)/(BORRADOR)_Analisis_con_Pandas_y_Kaggle_(Core).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# PROYECTO: ANÁLISIS DE VENTAS DE UNA TIENDA (dataset.csv de Kaggle)
# ============================================

# ============================================
# 1. PREPARACIÓN DEL ENTORNO
# ============================================
# Pandas ya viene instalado en Colab. Si hiciera falta: !pip install pandas
import pandas as pd
from google.colab import files
from IPython.display import display

print("Versión de Pandas:", pd.__version__)

# Aquí eliges el archivo dataset.csv desde tu computador
archivos = files.upload()
nombre_archivo = list(archivos.keys())[0]

# ============================================
# 2. CARGAR LOS DATOS
# ============================================
df = pd.read_csv(nombre_archivo)

print("\n--- Primeras 10 filas ---")
display(df.head(10))

# ============================================
# 3. EXPLORACIÓN INICIAL
# ============================================
print("\n--- Últimas 5 filas ---")
display(df.tail(5))

print("\n--- Información general ---")
df.info()

print("\n--- Estadísticas descriptivas (números) ---")
display(df.describe())

print("\n--- Estadísticas descriptivas (textos) ---")
display(df.describe(exclude="number"))

# ============================================
# 4. LIMPIEZA DE DATOS
# ============================================
# 4.1 Poner nombres de columnas en español, cortos y sin espacios (más fácil de escribir)
df = df.rename(columns={
    "Transaction ID": "ID_Transaccion",
    "Date": "Fecha",
    "Customer ID": "ID_Cliente",
    "Gender": "Genero",
    "Age": "Edad",
    "Product Category": "Categoria",
    "Quantity": "Cantidad",
    "Price per Unit": "Precio_Unitario",
    "Total Amount": "Total",
})

# 4.2 Corregir tipos de datos: la Fecha viene como texto y la pasamos a fecha real.
# Si alguna fecha estuviera mal escrita, queda vacía y la tratamos en el paso siguiente.
df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")

# 4.3 Datos faltantes: vemos cuántos hay en cada columna
print("\n--- Datos faltantes por columna ---")
print(df.isnull().sum())

# Si falta la fecha, eliminamos la fila (no podemos inventar una fecha).
df = df.dropna(subset=["Fecha"])

# Números faltantes: rellenamos con la mediana (no se deja influenciar por valores extremos).
for col in df.select_dtypes(include="number").columns:
    df[col] = df[col].fillna(df[col].median())

# Textos faltantes: rellenamos con el valor que más se repite.
for col in df.select_dtypes(exclude="number").columns:
    if col != "Fecha":
        df[col] = df[col].fillna(df[col].mode()[0])

# 4.4 Duplicados: filas idénticas que se repiten
print("\nFilas duplicadas encontradas:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

# 4.5 Revisión extra: comprobar que Total = Cantidad x Precio_Unitario
errores = (df["Total"] != df["Cantidad"] * df["Precio_Unitario"]).sum()
print("Filas donde el Total no cuadra con Cantidad x Precio:", errores)

print("Filas después de limpiar:", len(df))

# ============================================
# 5. TRANSFORMACIÓN DE DATOS
# ============================================
# 5.1 Columnas nuevas a partir de las que ya existen
df["Mes"] = df["Fecha"].dt.month                      # número del mes (1 a 12)
df["Nombre_Mes"] = df["Fecha"].dt.month_name()        # nombre del mes
df["Dia_Semana"] = df["Fecha"].dt.day_name()          # lunes, martes, etc.

# 5.2 Normalizar: dejar el Total entre 0 y 1 (0 = la compra más baja, 1 = la más alta)
minimo = df["Total"].min()
maximo = df["Total"].max()
df["Total_normalizado"] = ((df["Total"] - minimo) / (maximo - minimo)).round(3)

# 5.3 Estandarizar: cuántas "desviaciones" se aleja cada compra del promedio
df["Total_estandarizado"] = ((df["Total"] - df["Total"].mean()) / df["Total"].std()).round(3)

# 5.4 Clasificar en categorías
# a) Grupos de edad, con cortes que elegimos nosotros
df["Grupo_Edad"] = pd.cut(
    df["Edad"],
    bins=[0, 24, 39, 54, 120],
    labels=["Joven (hasta 24)", "Adulto joven (25-39)", "Adulto (40-54)", "Mayor (55+)"],
)
# b) Nivel de gasto: 3 grupos del mismo tamaño (los más bajos, medios y altos)
df["Nivel_Gasto"] = pd.qcut(df["Total"], q=3, labels=["Bajo", "Medio", "Alto"])

print("\n--- Datos con las columnas nuevas ---")
display(df.head())

# ============================================
# 6. ANÁLISIS DE DATOS (de lo más sencillo a lo más completo)
# ============================================

# --- Agrupar, ejemplo 1 (sencillo): ventas totales por categoría ---
print("\n--- Ventas totales por categoría ---")
display(df.groupby("Categoria")["Total"].sum().sort_values(ascending=False))

# --- Agrupar, ejemplo 2 (intermedio): varias cuentas a la vez por género ---
# sum=suma, mean=promedio, count=cantidad, min=mínimo, max=máximo,
# std=desviación estándar, var=varianza
print("\n--- Resumen de ventas por género ---")
display(df.groupby("Genero")["Total"].agg(["sum", "mean", "count", "min", "max", "std", "var"]).round(2))

# --- Agrupar, ejemplo 3 (completo): cruzar categoría y grupo de edad ---
print("\n--- Resumen por categoría y grupo de edad ---")
resumen = df.groupby(["Categoria", "Grupo_Edad"], observed=True).agg(
    Ventas_totales=("Total", "sum"),
    Compra_promedio=("Total", "mean"),
    Unidades=("Cantidad", "sum"),
    Compras=("Total", "count"),
).round(2)
display(resumen)

# Ventas por mes (útil para ver en qué época se vende más)
print("\n--- Ventas totales por mes ---")
display(df.groupby("Mes")["Total"].sum())

# --- Apply, ejemplo 1 (sencillo): escribir el género en español ---
df["Genero_ES"] = df["Genero"].apply(lambda g: "Hombre" if g == "Male" else "Mujer")

# --- Apply, ejemplo 2 (intermedio): clasificar cada compra con nuestra propia función ---
def clasificar_compra(total):
    if total >= 1000:
        return "Compra grande"
    elif total >= 200:
        return "Compra mediana"
    else:
        return "Compra pequeña"

df["Tipo_Compra"] = df["Total"].apply(clasificar_compra)

# --- Apply, ejemplo 3 (complejo): revisar la fila completa para sugerir una acción ---
def sugerir_accion(fila):
    if fila["Tipo_Compra"] == "Compra grande" and fila["Cantidad"] >= 3:
        return f"Ofrecer descuento por volumen en {fila['Categoria']}"
    elif fila["Tipo_Compra"] == "Compra pequeña" and fila["Edad"] < 30:
        return f"Enviar promoción de {fila['Categoria']} a clientes jóvenes"
    else:
        return "Sin cambios"

df["Accion_Sugerida"] = df.apply(sugerir_accion, axis=1)   # axis=1 = fila por fila

print("\n--- Resultado de los apply ---")
display(df[["Categoria", "Genero_ES", "Tipo_Compra", "Accion_Sugerida"]].head(10))

print("\n--- Cuántas veces sale cada acción sugerida ---")
print(df["Accion_Sugerida"].value_counts())

# ============================================
# 7. DOCUMENTACIÓN
# ============================================
# Resumen de lo que se hizo y por qué:
# - Cargamos el archivo y lo revisamos para conocer su forma, columnas y tipos de datos.
# - Renombramos las columnas a español para escribir el código más rápido y sin errores.
# - Convertimos Fecha a fecha real para poder sacar el mes y el día de la semana.
# - Revisamos datos faltantes y duplicados (en este archivo no había, pero el código
#   está listo por si trabajas con otro dataset que sí los tenga).
# - Comprobamos que Total = Cantidad x Precio_Unitario para confiar en los datos.
# - Creamos columnas nuevas (mes, grupo de edad, nivel de gasto) para poder comparar grupos.
# - Normalizamos y estandarizamos el Total para compararlo en una misma escala.
# - Agrupamos para ver qué categoría, género y edad venden más, y usamos apply
#   para aplicar reglas propias a cada compra.

Versión de Pandas: 2.2.3


Saving dataset.csv to dataset.csv

--- Primeras 10 filas ---


,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100
5,6,2023-04-25,CUST006,Female,45,Beauty,1,30,30
6,7,2023-03-13,CUST007,Male,46,Clothing,2,25,50
7,8,2023-02-22,CUST008,Male,30,Electronics,4,25,100
8,9,2023-12-13,CUST009,Male,63,Electronics,2,300,600
9,10,2023-10-07,CUST010,Female,52,Clothing,4,50,200



--- Últimas 5 filas ---


,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
995,996,2023-05-16,CUST996,Male,62,Clothing,1,50,50
996,997,2023-11-17,CUST997,Male,52,Beauty,3,30,90
997,998,2023-10-29,CUST998,Female,23,Beauty,4,25,100
998,999,2023-12-05,CUST999,Female,36,Electronics,3,50,150
999,1000,2023-04-12,CUST1000,Male,47,Electronics,4,30,120



--- Información general ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    1000 non-null   int64 
 1   Date              1000 non-null   object
 2   Customer ID       1000 non-null   object
 3   Gender            1000 non-null   object
 4   Age               1000 non-null   int64 
 5   Product Category  1000 non-null   object
 6   Quantity          1000 non-null   int64 
 7   Price per Unit    1000 non-null   int64 
 8   Total Amount      1000 non-null   int64 
dtypes: int64(5), object(4)
memory usage: 70.4+ KB

--- Estadísticas descriptivas (números) ---


... (12 outputs left out)

,Total
Mes,
1,36980
2,44060
3,28990
4,33870
5,53150
6,36715
7,35465
8,36960
9,23620



--- Resultado de los apply ---


,Categoria,Genero_ES,Tipo_Compra,Accion_Sugerida
0,Beauty,Hombre,Compra pequeña,Sin cambios
1,Clothing,Mujer,Compra grande,Sin cambios
2,Electronics,Hombre,Compra pequeña,Sin cambios
3,Clothing,Hombre,Compra mediana,Sin cambios
4,Beauty,Hombre,Compra pequeña,Sin cambios
5,Beauty,Mujer,Compra pequeña,Sin cambios
6,Clothing,Hombre,Compra pequeña,Sin cambios
7,Electronics,Hombre,Compra pequeña,Sin cambios
8,Electronics,Hombre,Compra mediana,Sin cambios
9,Clothing,Mujer,Compra mediana,Sin cambios



--- Cuántas veces sale cada acción sugerida ---
Accion_Sugerida
Sin cambios                                           718
Ofrecer descuento por volumen en Electronics           53
Ofrecer descuento por volumen en Clothing              52
Ofrecer descuento por volumen en Beauty                48
Enviar promoción de Beauty a clientes jóvenes          45
Enviar promoción de Clothing a clientes jóvenes        42
Enviar promoción de Electronics a clientes jóvenes     42
Name: count, dtype: int64
